### Phase 1 — Build KPIs

In [31]:
import pandas as pd
import numpy as np
import os

### Define input and output paths

In [32]:
input_path = "../../data/sample_synthetic_posts.csv"

output_folder = "../outputs"

output_path = "../outputs/kpi_dataset.csv"

# Create the output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

In [33]:
#load the data
df = pd.read_csv(input_path)

df.head()
print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns)
df.info()

Dataset shape: (300, 23)
Columns:
Index(['business_name', 'sector', 'followers_count', 'post_date',
       'posting_hour', 'day_of_week', 'month', 'post_type', 'caption_text',
       'caption_length', 'hashtags_count', 'emoji_count', 'likes_count',
       'comments_count', 'views_count', 'language', 'CTA_present',
       'promo_post', 'discount_percent', 'mentions_location',
       'religious_theme', 'patriotic_theme', 'arabic_dialect_style'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   business_name         300 non-null    str    
 1   sector                300 non-null    str    
 2   followers_count       300 non-null    int64  
 3   post_date             300 non-null    str    
 4   posting_hour          300 non-null    int64  
 5   day_of_week           300 non-null    str    
 6   month                 300 

### base of time series 

In [34]:
# Convert post_date from string to datetime format
# errors=coerce means invalid dates will become NaT instead of crashing the code
df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")

# Remove rows where post_date could not be converted
df = df.dropna(subset=["post_date"])

# Sort data by date because time-series needs chronological order
df = df.sort_values("post_date")

In [35]:
# We replace missing views with 0 because missing views should not break KPI calculations.
df["views_count"] = df["views_count"].fillna(0)

In [36]:
numeric_columns = [
    "followers_count",
    "likes_count",
    "comments_count",
    "views_count"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Replace any remaining missing numeric values with 0
df[numeric_columns] = df[numeric_columns].fillna(0)

### Create engagement KPI

In [37]:
# Engagement combines different types of interaction into one score.
# Likes are counted normally.
# Comments are multiplied by 2 because comments usually show stronger interest.
# Views are multiplied by 0.1 because views are weaker than likes/comments.

df["engagement"] = (
    df["likes_count"] +
    (df["comments_count"] * 2) +
    (df["views_count"] * 0.1)
)

### Create engagement_rate KPI


In [38]:

# Engagement rate normalizes engagement by the number of followers for helps compare small and large businesses fairly
# Replace 0 followers with NaN to avoid division by zero
df["followers_count_safe"] = df["followers_count"].replace(0, np.nan)
df["engagement_rate"] = df["engagement"] / df["followers_count_safe"]

# If engagement_rate is missing because followers were 0, replace it with 0
df["engagement_rate"] = df["engagement_rate"].fillna(0)

# We do not need this helper column anymore
df = df.drop(columns=["followers_count_safe"])

### Create week and month columns

In [39]:

# week groups posts by week.
# This will be used later to create weekly_trends.csv.
df["week"] = df["post_date"].dt.to_period("W").astype(str)


### Create post type indicators

In [40]:
# Normalize post_type text to avoid problems with uppercase/lowercase spaces
post_type_clean = df["post_type"].astype(str).str.lower().str.strip()

# is_video is True if the post type is video
df["is_video"] = post_type_clean.eq("video")

# is_reel is True if the post type is reel
df["is_reel"] = post_type_clean.eq("reel")

In [41]:
#  Preview the new KPI columns

df[
    [
        "business_name",
        "sector",
        "post_date",
        "post_type",
        "followers_count",
        "likes_count",
        "comments_count",
        "views_count",
        "engagement",
        "engagement_rate",
        "week",
        "is_video",
        "is_reel"
    ]
].head(10)

,business_name,sector,post_date,post_type,followers_count,likes_count,comments_count,views_count,engagement,engagement_rate,week,is_video,is_reel
274,Shifa Physiotherapy,Clinic,2025-06-03,image,10435,371,31,0.0,433.0,0.041495,2025-06-02/2025-06-08,False,False
251,Falafel Al-Balad,Restaurant,2025-06-05,image,11524,506,49,0.0,604.0,0.052412,2025-06-02/2025-06-08,False,False
13,Gaza Home Cleaning,Local Services,2025-06-05,carousel,2776,105,7,0.0,119.0,0.042867,2025-06-02/2025-06-08,False,False
167,Gaza Sea Breeze Cafe,Cafe,2025-06-06,reel,14649,520,39,14501.0,2048.1,0.139812,2025-06-02/2025-06-08,False,True
163,Cafe Al-Quds,Cafe,2025-06-07,carousel,12976,201,23,0.0,247.0,0.019035,2025-06-02/2025-06-08,False,False
236,Canaan Threads,Clothing Store,2025-06-07,reel,8964,265,16,15247.0,1821.7,0.203224,2025-06-02/2025-06-08,False,True
275,Falafel Al-Balad,Restaurant,2025-06-09,image,11524,340,24,0.0,388.0,0.033669,2025-06-09/2025-06-15,False,False
220,Ramallah Streetwear,Clothing Store,2025-06-09,image,18359,574,68,0.0,710.0,0.038673,2025-06-09/2025-06-15,False,False
179,Shifa Physiotherapy,Clinic,2025-06-13,image,10435,220,21,0.0,262.0,0.025108,2025-06-09/2025-06-15,False,False
138,Gaza Power House,Gym,2025-06-16,reel,1688,22,0,338.0,55.8,0.033057,2025-06-16/2025-06-22,False,True


### Validate the created KPIs

In [42]:
print("Missing values in KPI columns:")
print(df[["engagement", "engagement_rate", "week",  "is_video", "is_reel"]].isnull().sum())

print("\nKPI summary:")
print(df[["engagement", "engagement_rate"]].describe())

Missing values in KPI columns:
engagement         0
engagement_rate    0
week               0
is_video           0
is_reel            0
dtype: int64

KPI summary:
        engagement  engagement_rate
count   300.000000       300.000000
mean    563.635333         0.086758
std     699.616490         0.080513
min      11.000000         0.008376
25%     120.500000         0.031944
50%     333.500000         0.042561
75%     641.775000         0.142036
max    4140.000000         0.328199


In [43]:
#Save the KPI dataset
df.to_csv(output_path, index=False)

print("Phase 1 completed successfully.")
print("KPI dataset saved to:", output_path)
print("Final dataset shape:", df.shape)

Phase 1 completed successfully.
KPI dataset saved to: ../outputs/kpi_dataset.csv
Final dataset shape: (300, 28)


In [44]:
print(df.dtypes)

business_name                      str
sector                             str
followers_count                  int64
post_date               datetime64[us]
posting_hour                     int64
day_of_week                        str
month                            int64
post_type                          str
caption_text                       str
caption_length                   int64
hashtags_count                   int64
emoji_count                      int64
likes_count                      int64
comments_count                   int64
views_count                    float64
language                           str
CTA_present                       bool
promo_post                        bool
discount_percent               float64
mentions_location                 bool
religious_theme                   bool
patriotic_theme                   bool
arabic_dialect_style              bool
engagement                     float64
engagement_rate                float64
week                     

In [45]:
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,discount_percent,mentions_location,religious_theme,patriotic_theme,arabic_dialect_style,engagement,engagement_rate,week,is_video,is_reel
274,Shifa Physiotherapy,Clinic,10435,2025-06-03,17,Tuesday,6,image,"Support local, shop Palestinian. #Jerusalem #H...",76,...,0.0,False,False,False,False,433.0,0.041495,2025-06-02/2025-06-08,False,False
251,Falafel Al-Balad,Restaurant,11524,2025-06-05,20,Thursday,6,image,Limited offer for this week. Book now ?????,43,...,0.0,False,True,False,False,604.0,0.052412,2025-06-02/2025-06-08,False,False
13,Gaza Home Cleaning,Local Services,2776,2025-06-05,22,Thursday,6,carousel,Community vibes in Ramallah. ?????,34,...,0.0,True,False,False,False,119.0,0.042867,2025-06-02/2025-06-08,False,False
167,Gaza Sea Breeze Cafe,Cafe,14649,2025-06-06,21,Friday,6,reel,Customer favorite is back. Visit us today | ah...,111,...,4.0,False,False,False,True,2048.1,0.139812,2025-06-02/2025-06-08,False,True
163,Cafe Al-Quds,Cafe,12976,2025-06-07,3,Saturday,6,carousel,"Support local, shop Palestinian. Order on What...",66,...,24.0,False,False,True,False,247.0,0.019035,2025-06-02/2025-06-08,False,False
